# DipRadar v3.1 — Treino com `alpha_60d`, 10 folds + purga, 4 features novas e calibrator

**O que muda face ao bundle v3 actual** (`dip_models_v3.pkl`):

1. **Target → `alpha_60d`** = `max_return_60d − spy_max_return_60d` no mesmo período.
   O modelo deixa de aprender beta de mercado.

2. **CV → 10 folds expanding-window com purge gap 21 dias.**

3. **+4 features novas**: `relative_drop`, `sector_alert_count_7d`,
   `days_since_52w_high`, `month_of_year`.

4. **Score calibrator (Isotonic)** treinado em predições out-of-fold,
   anexado ao bundle no campo `score_calibrator`.

5. **Sample weights temporais** com half-life 3 anos
   (vs 1.5 anos anteriores) — mantém peso meaningful nos regimes 2008/2011/2015.

> ⚠ **Antes de fazer deploy**: 2 das 4 features novas exigem trabalho do lado da
> inferência (`sector_alert_count_7d` precisa de leitura do `alert_db.csv` em
> runtime; `days_since_52w_high` precisa de `price_history`). Sem isso, em
> produção essas features caem para `0.0` (semanticamente errado). As outras 2
> (`relative_drop`, `month_of_year`) são triviais.
> Detalhes em `docs/retrain_plan_v3_1.md` § 3.

Ver `docs/retrain_plan_v3_1.md` para detalhes completos das mudanças.

**Como correr**: Runtime → Run all. Demora ~10-20 min consoante o batch
de fetch yfinance.

## 0. Setup — clone + imports

In [ ]:
# ── Clone / pull repo (Colab) ou usar repo local ───────────────────────
import os, sys
from pathlib import Path

REPO_URL  = 'https://github.com/romeurf/DipRadar.git'
REPO_NAME = 'DipRadar'

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if Path(REPO_NAME).exists():
        print('Repo já existe — a fazer pull...')
        os.system(f'cd {REPO_NAME} && git pull -q')
    else:
        print('A clonar repo...')
        os.system(f'git clone -q {REPO_URL}')
    os.chdir(REPO_NAME)

assert Path('ml_features.py').exists(), (
    'ml_features.py não encontrado. Corre o notebook a partir da raiz do repo.'
)

REPO_ROOT = Path.cwd()
for p in (REPO_ROOT, REPO_ROOT / 'experiments' / 'ml_v2'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f'REPO_ROOT : {REPO_ROOT}')
os.system('git log --oneline -1')

In [ ]:
# ── Instalar deps que o Colab não traz por defeito ─────────────────────
# (Se já tiveres o ambiente pronto, esta célula é instantânea.)
import importlib

for pkg, pip_name in [
    ('xgboost',   'xgboost'),
    ('lightgbm',  'lightgbm'),
    ('yfinance',  'yfinance'),
    ('sklearn',   'scikit-learn'),
]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        os.system(f'pip install -q {pip_name}')

print('Deps OK')

In [ ]:
# ── Imports + configuração geral ───────────────────────────────────────
import warnings, logging, math, pickle, json
from datetime import datetime, date
from dataclasses import dataclass, field, fields, is_dataclass
from typing import Any, Optional

import numpy as np
import pandas as pd

from sklearn.ensemble        import RandomForestRegressor
from sklearn.isotonic        import IsotonicRegression
from sklearn.linear_model    import Ridge
from sklearn.metrics         import brier_score_loss
from xgboost                 import XGBRegressor
from lightgbm                import LGBMRegressor

from scipy.stats import spearmanr

import yfinance as yf

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(message)s')

# Paths
DATA_DIR    = REPO_ROOT
ALERT_DB    = DATA_DIR / 'alert_db.csv'
PARQUET_V1  = DATA_DIR / 'ml_training_merged.parquet'
PARQUET_V31 = REPO_ROOT / 'experiments/ml_v2/dataset_v31.parquet'
MODELS_DIR  = REPO_ROOT / 'experiments/ml_v2'
MODEL_OUT   = REPO_ROOT / 'dip_models_v3.pkl'           # mantém o nome do bundle
REPORT_OUT  = REPO_ROOT / 'ml_report_v3.json'

HORIZON_DAYS  = 60
WINSOR_PCT    = 0.01
HALF_LIFE_DAYS = 365 * 3   # sample weights — half-life 3 anos
N_FOLDS       = 10
PURGE_DAYS    = 21
TOPK_FRAC     = 0.20

print(f'PARQUET_V1 : {PARQUET_V1}  exists={PARQUET_V1.exists()}')
print(f'MODEL_OUT  : {MODEL_OUT}')
print(f'REPORT_OUT : {REPORT_OUT}')

## 1. Carregar dataset v1/v2 + alert_db.csv

In [ ]:
# ── Dataset base (parquet com todas as features v1/v2) ────────────────
df = pd.read_parquet(PARQUET_V1)
df['alert_date'] = pd.to_datetime(df['alert_date'])
df = df.sort_values('alert_date').reset_index(drop=True)

# Compatibilidade de nomes de colunas
if 'symbol' in df.columns and 'ticker' not in df.columns:
    df = df.rename(columns={'symbol': 'ticker'})

print(f'Shape: {df.shape}')
print(f'Período: {df["alert_date"].min().date()} → {df["alert_date"].max().date()}')
print(f'Tickers únicos: {df["ticker"].nunique()}')
print()

# Cobertura de targets já resolvidos
for col in ['max_return_60d', 'max_drawdown_60d', 'spy_return_ref']:
    if col in df.columns:
        n = df[col].notna().sum()
        pct = df[col].notna().mean()
        print(f'  {col:20s}: {n:5d} / {len(df)} ({pct:.1%})')

In [ ]:
# ── Mergear alertas mais recentes (alert_db.csv), só para registo ─────
# Os que não têm target ficam fora do treino; servem para EDA.
if ALERT_DB.exists():
    df_adb = pd.read_csv(ALERT_DB, parse_dates=['alert_date'])
    if 'symbol' in df_adb.columns and 'ticker' not in df_adb.columns:
        df_adb = df_adb.rename(columns={'symbol': 'ticker'})

    existing = set(zip(df['ticker'], df['alert_date'].dt.date))
    mask_new = ~df_adb.apply(
        lambda r: (r['ticker'], pd.Timestamp(r['alert_date']).date()) in existing,
        axis=1,
    )
    n_new = int(mask_new.sum())
    print(f'alert_db.csv: {len(df_adb)} alertas | novos sem target: {n_new}')
else:
    print('alert_db.csv não encontrado — sem alertas extra a juntar.')

## 2. Fetch de price_history (SPY + sector ETFs + stocks)

Vamos buscar OHLCV de SPY (para o target alpha), do ETF sectorial de cada
sector (`add_momentum_features`), e do ticker individual (price-based features
v2 + alpha).

In [ ]:
# ── Sector → ETF (cópia local de macro_data.SECTOR_ETF) ────────────────
SECTOR_ETF = {
    'Technology':            'XLK',
    'Financial Services':    'XLF',
    'Healthcare':            'XLV',
    'Consumer Cyclical':     'XLY',
    'Consumer Defensive':    'XLP',
    'Industrials':           'XLI',
    'Energy':                'XLE',
    'Utilities':             'XLU',
    'Real Estate':           'XLRE',
    'Basic Materials':       'XLB',
    'Communication Services':'XLC',
    'Unknown':               'SPY',
}
DEFAULT_ETF = 'SPY'

# Tickers a fetchar
tickers   = sorted(df['ticker'].dropna().unique().tolist())
etfs      = sorted({DEFAULT_ETF} | {SECTOR_ETF.get(s, DEFAULT_ETF) for s in df['sector'].dropna().unique()})

print(f'A fetchar: {len(tickers)} stocks + {len(etfs)} ETFs (SPY incluído)')

In [ ]:
# ── Fetch em batch (yfinance) ─────────────────────────────────────────
# Cobertura: desde 5 anos antes da primeira alert até hoje.
START = (df['alert_date'].min() - pd.Timedelta(days=365 * 5)).strftime('%Y-%m-%d')
END   = (df['alert_date'].max() + pd.Timedelta(days=HORIZON_DAYS + 7)).strftime('%Y-%m-%d')

def _fetch_ohlcv_batch(tickers_list, start, end, batch_size=40):
    out = {}
    for i in range(0, len(tickers_list), batch_size):
        batch = tickers_list[i:i + batch_size]
        try:
            data = yf.download(
                batch, start=start, end=end,
                progress=False, group_by='ticker',
                auto_adjust=False, threads=True,
            )
        except Exception as e:
            print(f'  batch {i}-{i+batch_size}: erro {e!r}')
            continue
        for tk in batch:
            try:
                if len(batch) == 1:
                    sub = data[['Open','High','Low','Close','Volume']].dropna(subset=['Close'])
                else:
                    sub = data[tk][['Open','High','Low','Close','Volume']].dropna(subset=['Close'])
                if len(sub) > 50:
                    out[tk] = sub
            except Exception:
                pass
        print(f'  fetched {min(i + batch_size, len(tickers_list))}/{len(tickers_list)}')
    return out

print('Fetching ETFs...')
etf_cache = _fetch_ohlcv_batch(etfs, START, END, batch_size=20)
print(f'  ETFs OK: {len(etf_cache)}/{len(etfs)}')

print('Fetching stocks...')
price_cache = _fetch_ohlcv_batch(tickers, START, END, batch_size=40)
print(f'  Stocks OK: {len(price_cache)}/{len(tickers)}')

## 3. Construir `dataset_v31` com features v2 + momentum + 4 NEW + target alpha

Cada linha do parquet vira uma linha do dataset enriquecido. Para cada alerta:

- Recolhe features v1/v2 (já no parquet) + momentum (Stage 3b)
- Calcula 4 features NEW: `relative_drop`, `sector_alert_count_7d`,
  `days_since_52w_high`, `month_of_year`
- Recalcula `max_return_60d` e `max_drawdown_60d` a partir do price_history
  (anti-leakage)
- Calcula `spy_max_return_60d` a partir do SPY no mesmo período
- `alpha_60d = max_return_60d - spy_max_return_60d`

In [ ]:
# ── Importar helpers do repo (ml_features tem add_derived/momentum) ───
from ml_features import (
    FEATURE_COLUMNS, _FALLBACK,
    add_derived_features, add_momentum_features,
)
from experiments.ml_v2.pipeline import (
    FEATURE_COLUMNS_V2, build_v2_features, build_targets,
)

MOMENTUM_FEATURES = ['return_1m', 'return_3m_pre', 'sector_relative', 'beta_60d']
NEW_FEATURES      = ['relative_drop', 'sector_alert_count_7d',
                     'days_since_52w_high', 'month_of_year']

# Dedup: FEATURE_COLUMNS já inclui as 4 momentum, e FEATURE_COLUMNS_V2
# inclui FEATURE_COLUMNS — então só temos de adicionar as NEW.
_seen = set()
FEATURE_COLUMNS_V31 = []
for c in list(FEATURE_COLUMNS_V2) + list(MOMENTUM_FEATURES) + list(NEW_FEATURES):
    if c not in _seen:
        _seen.add(c); FEATURE_COLUMNS_V31.append(c)
print(f'Total features v31: {len(FEATURE_COLUMNS_V31)} '
      f'(union of V2 + momentum + new, deduplicado)')

In [ ]:
# ── Pré-calcular sector_alert_count_7d (rolling sobre alert_date) ─────
df_with_sec = df[['alert_date', 'sector', 'ticker']].copy()
df_with_sec = df_with_sec.set_index('alert_date').sort_index()

# Para cada alerta, contar quantos do mesmo sector ocorreram nos 7 dias antes
# (não inclui o próprio alerta). Anti-leakage: só olha para trás.
def _count_7d(group):
    counts = []
    dates = group.index.to_list()
    for i, d in enumerate(dates):
        window_start = d - pd.Timedelta(days=7)
        # count alerts in [window_start, d) — strict < d
        cnt = sum(1 for dd in dates[:i] if window_start <= dd < d)
        counts.append(cnt)
    return pd.Series(counts, index=group.index)

df_with_sec['sector_alert_count_7d'] = (
    df_with_sec.groupby('sector', group_keys=False)
               .apply(lambda g: _count_7d(g))
)
sector_count_lookup = df_with_sec.reset_index().set_index(
    ['ticker', 'alert_date']
)['sector_alert_count_7d'].to_dict()
print(f'sector_alert_count_7d: {len(sector_count_lookup)} pares (ticker,alert_date) com contagem')

In [ ]:
# ── Construir dataset_v31 linha-a-linha ────────────────────────────────
def _spy_max_return_60d(spy_hist, alert_date, horizon=HORIZON_DAYS):
    """Forward-only SPY max return em [alert_date+1, alert_date+horizon]."""
    if spy_hist is None:
        return np.nan
    entry_slice = spy_hist[spy_hist.index <= alert_date]
    if len(entry_slice) == 0:
        return np.nan
    spy_entry = float(entry_slice['Close'].iloc[-1])
    if spy_entry <= 0:
        return np.nan
    fwd = spy_hist[(spy_hist.index > alert_date) &
                   (spy_hist.index <= alert_date + pd.Timedelta(days=horizon))]
    if len(fwd) < 5:
        return np.nan
    return float(fwd['Close'].max() / spy_entry - 1.0)


def _days_since_52w_high(hist, alert_date):
    """Quantos dias passaram desde o pico de 52 semanas (até alert_date)."""
    window = hist[(hist.index <= alert_date) &
                  (hist.index >  alert_date - pd.Timedelta(days=365))]
    if len(window) < 20:
        return _FALLBACK.get('days_since_52w_high', 60.0)
    high_idx = window['High'].idxmax()
    return float((alert_date - high_idx).days)


spy_hist = etf_cache.get('SPY')
rows_v31 = []
skipped  = {'no_price': 0, 'short_history': 0, 'no_target': 0, 'no_spy_target': 0}

for _, row in df.iterrows():
    ticker     = row['ticker']
    alert_date = pd.Timestamp(row['alert_date'])
    sector     = row.get('sector', 'Unknown') or 'Unknown'
    etf        = SECTOR_ETF.get(sector, DEFAULT_ETF)

    ohlcv = price_cache.get(ticker)
    if ohlcv is None:
        skipped['no_price'] += 1; continue

    hist = ohlcv[ohlcv.index <= alert_date]
    if len(hist) < 25:
        skipped['short_history'] += 1; continue

    # Features v1 (do parquet, com fallback)
    fv = {c: float(row[c]) if (c in row.index and pd.notna(row.get(c))) else _FALLBACK.get(c, 0.0)
          for c in FEATURE_COLUMNS}
    add_derived_features(fv)

    # Features v2 extra (price-based)
    fv.update(build_v2_features(row, hist))

    # Features v3 (momentum)
    sec_hist = etf_cache.get(etf)
    sec_slice = sec_hist[sec_hist.index <= alert_date] if sec_hist is not None else None
    spy_slice = spy_hist[spy_hist.index <= alert_date] if spy_hist is not None else None
    add_momentum_features(fv, hist, sec_slice, spy_slice)

    # Features v31 NEW
    fv['relative_drop']         = float(fv.get('drop_pct_today', 0.0)
                                        - fv.get('sector_drawdown_5d', 0.0))
    fv['sector_alert_count_7d'] = float(sector_count_lookup.get((ticker, alert_date), 0))
    fv['days_since_52w_high']   = _days_since_52w_high(ohlcv, alert_date)
    fv['month_of_year']         = float(alert_date.month)

    # Target real do parquet (se tiver) — caso contrário recalcular
    if 'max_return_60d' in row.index and pd.notna(row.get('max_return_60d')):
        max_ret  = float(row['max_return_60d'])
        max_draw = float(row.get('max_drawdown_60d', 0.0))
    else:
        entry_price = float(row.get('price', 0.0))
        if entry_price <= 0:
            skipped['no_target'] += 1; continue
        future = ohlcv[(ohlcv.index > alert_date) &
                       (ohlcv.index <= alert_date + pd.Timedelta(days=HORIZON_DAYS))]['Close']
        if len(future) < 5:
            skipped['no_target'] += 1; continue
        tgt = build_targets(alert_date, entry_price, future)
        if math.isnan(tgt['max_return_60d']):
            skipped['no_target'] += 1; continue
        max_ret  = tgt['max_return_60d']
        max_draw = tgt['max_drawdown_60d']

    spy_max_ret = _spy_max_return_60d(spy_hist, alert_date)
    if math.isnan(spy_max_ret):
        skipped['no_spy_target'] += 1; continue

    alpha_60d = max_ret - spy_max_ret

    rec = {
        'ticker':     ticker,
        'alert_date': alert_date,
        'sector':     sector,
        **{c: fv[c] for c in FEATURE_COLUMNS_V31 if c in fv},
        'max_return_60d':     max_ret,
        'max_drawdown_60d':   max_draw,
        'spy_max_return_60d': spy_max_ret,
        'alpha_60d':          alpha_60d,
    }
    rows_v31.append(rec)

df_v31 = pd.DataFrame(rows_v31)
print(f'\nDataset v3.1 shape: {df_v31.shape}')
print(f'Skipped: {skipped}')
print(f'\nDistribuição alpha_60d:')
print(df_v31['alpha_60d'].describe().round(4))
print(f'\n% alertas com alpha > 5%: {(df_v31["alpha_60d"] > 0.05).mean():.1%}')
print(f'% alertas com alpha > 0%:  {(df_v31["alpha_60d"] > 0.00).mean():.1%}')

In [ ]:
# ── Persistir dataset_v31 (cache para iterações rápidas) ──────────────
df_v31.to_parquet(PARQUET_V31, index=False)
print(f'Dataset guardado: {PARQUET_V31} ({PARQUET_V31.stat().st_size / 1024:.1f} KB)')

## 4. EDA — sanity checks das novas features e do target alpha

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 1) Distribuição alpha_60d
axes[0, 0].hist(df_v31['alpha_60d'].clip(-0.4, 0.6), bins=50, edgecolor='k')
axes[0, 0].axvline(0,    color='k', linestyle='--', alpha=0.5)
axes[0, 0].axvline(0.05, color='g', linestyle='--', alpha=0.7, label='+5%')
axes[0, 0].set_title('alpha_60d (clip [-0.4, 0.6])')
axes[0, 0].legend()

# 2) Comparar alpha vs max_return (mostra "deflação" do beta)
axes[0, 1].scatter(
    df_v31['max_return_60d'].clip(-0.4, 1.0),
    df_v31['alpha_60d'].clip(-0.4, 0.6),
    s=4, alpha=0.3,
)
axes[0, 1].axline((0, 0), slope=1, color='r', linestyle='--', alpha=0.5, label='y = x')
axes[0, 1].axline((0, 0), slope=0, color='k', linestyle='--', alpha=0.3)
axes[0, 1].set_xlabel('max_return_60d (absolute)')
axes[0, 1].set_ylabel('alpha_60d (vs SPY)')
axes[0, 1].set_title('Alpha removido beta de mercado')
axes[0, 1].legend()

# 3) relative_drop distribution
axes[0, 2].hist(df_v31['relative_drop'].clip(-0.20, 0.10), bins=50, edgecolor='k')
axes[0, 2].set_title('relative_drop = drop_today − sector_5d')

# 4) sector_alert_count_7d
axes[1, 0].hist(df_v31['sector_alert_count_7d'].clip(0, 30), bins=30, edgecolor='k')
axes[1, 0].set_title('sector_alert_count_7d')

# 5) days_since_52w_high
axes[1, 1].hist(df_v31['days_since_52w_high'].clip(0, 365), bins=40, edgecolor='k')
axes[1, 1].set_title('days_since_52w_high')

# 6) month_of_year
month_counts = df_v31['month_of_year'].value_counts().sort_index()
axes[1, 2].bar(month_counts.index, month_counts.values)
axes[1, 2].set_title('month_of_year (count of alerts)')
axes[1, 2].set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlação das features com alpha (top 15) ────────────────────────
corrs = []
for c in FEATURE_COLUMNS_V31:
    if c in df_v31.columns:
        corrs.append((c, float(df_v31[c].corr(df_v31['alpha_60d']))))
corrs.sort(key=lambda x: abs(x[1]), reverse=True)
print('Top 15 features por |corr(feature, alpha_60d)|:')
for name, r in corrs[:15]:
    bar = '█' * int(abs(r) * 80)
    sign = '+' if r >= 0 else '-'
    print(f'  {name:25s} {sign}{abs(r):.3f}  {bar}')

## 5. Walk-Forward CV — 10 folds expanding-window, purge gap 21 dias

Cada fold:
- treino = todas as observações até `train_end_k`
- purga 21 dias (descartadas)
- teste = obs em `(train_end_k + 21d, test_end_k]`

In [ ]:
# ── Setup folds + sample weights temporais ────────────────────────────
df_v31 = df_v31.sort_values('alert_date').reset_index(drop=True)
df_v31['alert_date'] = pd.to_datetime(df_v31['alert_date'])

min_date  = df_v31['alert_date'].min()
max_date  = df_v31['alert_date'].max()
total_days = (max_date - min_date).days

# expanding-window folds: cada fold k cobre fracção (0.5 + k/(2*N_FOLDS)) do período
fold_specs = []
for k in range(N_FOLDS):
    train_frac = 0.50 + 0.50 * k / N_FOLDS
    test_frac  = 0.50 + 0.50 * (k + 1) / N_FOLDS
    train_end  = min_date + pd.Timedelta(days=int(total_days * train_frac))
    purge_end  = train_end + pd.Timedelta(days=PURGE_DAYS)
    test_end   = min_date + pd.Timedelta(days=int(total_days * test_frac))
    if test_end <= purge_end:
        # demasiado pequeno; salta
        continue
    fold_specs.append((k + 1, train_end, purge_end, test_end))

print(f'{len(fold_specs)} folds construídos:')
for k, tr, pg, te in fold_specs:
    n_tr = (df_v31['alert_date'] <= tr).sum()
    n_te = ((df_v31['alert_date'] > pg) & (df_v31['alert_date'] <= te)).sum()
    print(f'  Fold {k:2d}: train≤{tr.date()} | purge {PURGE_DAYS}d | test ({pg.date()}, {te.date()}]'
          f'  → tr={n_tr} te={n_te}')

In [ ]:
# ── Helpers de avaliação ──────────────────────────────────────────────
def winsorize(arr, pct=WINSOR_PCT):
    if len(arr) == 0:
        return arr
    lo = np.quantile(arr, pct)
    hi = np.quantile(arr, 1 - pct)
    return np.clip(arr, lo, hi)


def spearman_safe(pred, true):
    pred = np.asarray(pred); true = np.asarray(true)
    mask = np.isfinite(pred) & np.isfinite(true)
    if mask.sum() < 5:
        return float('nan')
    rho, _ = spearmanr(pred[mask], true[mask])
    return float(rho)


def topk_pnl(pred_alpha, true_alpha, k=TOPK_FRAC):
    pred  = np.asarray(pred_alpha); true = np.asarray(true_alpha)
    mask  = np.isfinite(pred) & np.isfinite(true)
    pred  = pred[mask]; true = true[mask]
    if len(pred) < 5:
        return float('nan')
    n_top = max(1, int(len(pred) * k))
    top   = np.argsort(-pred)[:n_top]
    return float(true[top].mean())


def temporal_weights(alert_dates, max_date):
    dt   = pd.to_datetime(alert_dates)
    days = (max_date - dt).dt.days.values.astype(float)
    return np.exp(-np.log(2) * days / HALF_LIFE_DAYS)

In [ ]:
# ── Modelos candidatos ───────────────────────────────────────────────
def _xgb_factory():
    return XGBRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42,
        tree_method='hist', verbosity=0,
    )

def _lgbm_factory():
    return LGBMRegressor(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42,
        verbosity=-1,
    )

def _rf_factory():
    return RandomForestRegressor(
        n_estimators=300, max_depth=8, min_samples_leaf=10,
        n_jobs=-1, random_state=42,
    )

def _ridge_factory():
    return Ridge(alpha=1.0, random_state=42)


MODEL_CONFIGS = {
    'XGB-alpha-v31':   {'factory': _xgb_factory,   'feats': FEATURE_COLUMNS_V31},
    'LGBM-alpha-v31':  {'factory': _lgbm_factory,  'feats': FEATURE_COLUMNS_V31},
    'RF-alpha-v31':    {'factory': _rf_factory,    'feats': FEATURE_COLUMNS_V31},
    'Ridge-alpha-v31': {'factory': _ridge_factory, 'feats': FEATURE_COLUMNS_V31},
    # Baseline: XGB sem as 4 NEW features (controlo de impacto)
    'XGB-alpha-baseline': {'factory': _xgb_factory, 'feats': FEATURE_COLUMNS_V2 + MOMENTUM_FEATURES},
}
print(f'Modelos a comparar: {list(MODEL_CONFIGS.keys())}')

In [ ]:
# ── Walk-forward CV principal ────────────────────────────────────────
results = {name: [] for name in MODEL_CONFIGS}
oof_pred = {name: np.full(len(df_v31), np.nan) for name in MODEL_CONFIGS}

for k, train_end, purge_end, test_end in fold_specs:
    tr_mask = df_v31['alert_date'] <= train_end
    te_mask = (df_v31['alert_date'] > purge_end) & (df_v31['alert_date'] <= test_end)

    df_tr = df_v31[tr_mask]
    df_te = df_v31[te_mask]
    if len(df_tr) < 100 or len(df_te) < 20:
        print(f'Fold {k}: insuficiente (tr={len(df_tr)}, te={len(df_te)}) — saltar'); continue

    y_alpha_tr = winsorize(df_tr['alpha_60d'].values)
    y_alpha_te = df_te['alpha_60d'].values
    y_down_tr  = winsorize(df_tr['max_drawdown_60d'].values)
    y_down_te  = df_te['max_drawdown_60d'].values

    sw_tr = temporal_weights(df_tr['alert_date'], max_date)

    for name, cfg in MODEL_CONFIGS.items():
        feats = [f for f in cfg['feats'] if f in df_tr.columns]
        X_tr = df_tr[feats].fillna(0).values.astype(np.float32)
        X_te = df_te[feats].fillna(0).values.astype(np.float32)

        m_alpha = cfg['factory']()
        try:
            m_alpha.fit(X_tr, y_alpha_tr, sample_weight=sw_tr)
        except TypeError:
            # ridge não aceita sample_weight kwarg igual? aceita — esta linha defensiva
            m_alpha.fit(X_tr, y_alpha_tr)

        m_down = cfg['factory']()
        try:
            m_down.fit(X_tr, y_down_tr, sample_weight=sw_tr)
        except TypeError:
            m_down.fit(X_tr, y_down_tr)

        pred_alpha = m_alpha.predict(X_te)
        pred_down  = m_down.predict(X_te)

        oof_pred[name][df_te.index.values] = pred_alpha

        rho_alpha = spearman_safe(pred_alpha, y_alpha_te)
        rho_down  = spearman_safe(pred_down,  y_down_te)
        pnl       = topk_pnl(pred_alpha, y_alpha_te)

        results[name].append({
            'fold':       k,
            'n_test':     len(df_te),
            'rho_alpha':  rho_alpha,
            'rho_down':   rho_down,
            'rho_mean':   (rho_alpha - rho_down) / 2 if math.isfinite(rho_alpha) and math.isfinite(rho_down) else rho_alpha,
            'topk_pnl':   pnl,
        })

    print(f'Fold {k:2d} OK — n_train={len(df_tr)} n_test={len(df_te)}')

print('\nWalk-forward CV concluído.')

## 6. Resultados — comparação dos modelos

In [ ]:
# ── Tabela resumo (média e std por fold) ──────────────────────────────
summary_rows = []
for name, hist in results.items():
    if not hist:
        continue
    rho_alphas = np.array([h['rho_alpha'] for h in hist if math.isfinite(h['rho_alpha'])])
    rho_downs  = np.array([h['rho_down']  for h in hist if math.isfinite(h['rho_down'])])
    pnls       = np.array([h['topk_pnl']  for h in hist if math.isfinite(h['topk_pnl'])])
    summary_rows.append({
        'model':         name,
        'rho_alpha_mean': rho_alphas.mean() if len(rho_alphas) else np.nan,
        'rho_alpha_std':  rho_alphas.std()  if len(rho_alphas) else np.nan,
        'rho_down_mean':  rho_downs.mean()  if len(rho_downs) else np.nan,
        'topk_pnl_mean':  pnls.mean()       if len(pnls) else np.nan,
        'topk_pnl_std':   pnls.std()        if len(pnls) else np.nan,
        'n_folds':        len(hist),
    })
summary = pd.DataFrame(summary_rows).sort_values('rho_alpha_mean', ascending=False)
print(summary.round(4).to_string(index=False))

In [ ]:
# ── Plot por fold (estabilidade temporal) ─────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for name, hist in results.items():
    if not hist: continue
    folds = [h['fold'] for h in hist]
    ax[0].plot(folds, [h['rho_alpha'] for h in hist], marker='o', label=name)
    ax[1].plot(folds, [h['topk_pnl']  for h in hist], marker='o', label=name)
ax[0].axhline(0,    color='k', linestyle='--', alpha=0.4)
ax[0].axhline(0.30, color='g', linestyle='--', alpha=0.4, label='target rho≥0.30')
ax[0].set_xlabel('fold'); ax[0].set_ylabel('rho_alpha'); ax[0].set_title('Spearman ρ (alpha) por fold')
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].axhline(0,    color='k', linestyle='--', alpha=0.4)
ax[1].set_xlabel('fold'); ax[1].set_ylabel('topk_pnl (alpha)'); ax[1].set_title('Top-K PnL (alpha) por fold')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Champion — selecção, treino full + score_calibrator

In [ ]:
# ── Champion = melhor rho_alpha_mean com PnL > 0 ─────────────────────
qualifiers = summary[(summary['topk_pnl_mean'] > 0)].sort_values(
    'rho_alpha_mean', ascending=False
)
if len(qualifiers) == 0:
    print('AVISO: nenhum modelo passou no critério topk_pnl > 0')
    qualifiers = summary.sort_values('rho_alpha_mean', ascending=False)

CHAMPION_NAME = qualifiers.iloc[0]['model']
print(f'Champion: {CHAMPION_NAME}')
print(qualifiers.iloc[0].round(4).to_string())

In [ ]:
# ── Calibrator isotónico em OOF ───────────────────────────────────────
# Treina IsotonicRegression(y_oof_alpha → P(alpha > 0.05))
oof = oof_pred[CHAMPION_NAME]
mask = np.isfinite(oof)
y_oof = oof[mask]
y_bin = (df_v31['alpha_60d'].values[mask] > 0.05).astype(int)

print(f'OOF samples: {mask.sum()} / {len(df_v31)}')
print(f'Positive rate (alpha > 5%): {y_bin.mean():.1%}')

iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
iso.fit(y_oof, y_bin)

# Brier score (calibrar bem ≠ ranking bem)
prob_oof = iso.predict(y_oof)
brier    = brier_score_loss(y_bin, prob_oof)
print(f'Brier score OOF: {brier:.4f}  (target < 0.20)')

# Mostrar a curva de calibração mapeada
xs = np.linspace(np.quantile(y_oof, 0.01), np.quantile(y_oof, 0.99), 50)
ys = iso.predict(xs)
plt.figure(figsize=(7, 4))
plt.plot(xs, ys, 'o-', label='IsotonicRegression(OOF)')
plt.axvline(0.05, color='g', linestyle='--', alpha=0.5, label='α=5% (win)')
plt.axhline(0.50, color='k', linestyle='--', alpha=0.3)
plt.xlabel('predicted alpha (raw score)')
plt.ylabel('P(alpha > 5%)')
plt.title('Score calibrator — mapeamento alpha → probabilidade')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# ── Treinar champion no dataset COMPLETO ──────────────────────────────
champion_cfg   = MODEL_CONFIGS[CHAMPION_NAME]
champion_feats = [f for f in champion_cfg['feats'] if f in df_v31.columns]

X_full       = df_v31[champion_feats].fillna(0).values.astype(np.float32)
y_alpha_full = winsorize(df_v31['alpha_60d'].values)
y_down_full  = winsorize(df_v31['max_drawdown_60d'].values)
sw_full      = temporal_weights(df_v31['alert_date'], max_date)

champ_alpha = champion_cfg['factory']()
champ_down  = champion_cfg['factory']()

try:    champ_alpha.fit(X_full, y_alpha_full, sample_weight=sw_full)
except TypeError: champ_alpha.fit(X_full, y_alpha_full)
try:    champ_down.fit(X_full,  y_down_full,  sample_weight=sw_full)
except TypeError: champ_down.fit(X_full, y_down_full)

print(f'Champion treinado em {len(X_full)} amostras com {len(champion_feats)} features')

## 8. Empacotar `dip_models_v3.pkl` (DipModelsV3 com `score_calibrator`)

In [ ]:
# ── Definir DipModelsV3 (matches ml_predictor.py) ────────────────────
# Mantida no namespace __main__ para o pickle resolver na produção (Railway).
@dataclass
class DipModelsV3:
    model_up:        Any
    model_down:      Any
    feature_cols:    list
    score_calibrator: Any = None
    n_train_samples: int  = 0
    train_date:      str  = ''
    champion_name:   str  = ''
    schema_version:  int  = 3
    momentum_feats:  list = field(default_factory=list)
    rho_mean:        Optional[float] = None
    rho_alpha:       Optional[float] = None
    rho_down:        Optional[float] = None
    topk_pnl:        Optional[float] = None
    fold_metrics:    list = field(default_factory=list)


bundle = DipModelsV3(
    model_up         = champ_alpha,
    model_down       = champ_down,
    feature_cols     = champion_feats,
    score_calibrator = iso,
    n_train_samples  = int(len(X_full)),
    train_date       = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    champion_name    = CHAMPION_NAME,
    schema_version   = 3,
    momentum_feats   = MOMENTUM_FEATURES,
    rho_mean         = float(qualifiers.iloc[0]['rho_alpha_mean']),
    rho_alpha        = float(qualifiers.iloc[0]['rho_alpha_mean']),
    rho_down         = float(qualifiers.iloc[0]['rho_down_mean']),
    topk_pnl         = float(qualifiers.iloc[0]['topk_pnl_mean']),
    fold_metrics     = results[CHAMPION_NAME],
)

# Pickle (joblib é mais robusto para árvores grandes)
import joblib
joblib.dump(bundle, MODEL_OUT)
print(f'Bundle escrito: {MODEL_OUT} ({MODEL_OUT.stat().st_size / 1024:.1f} KB)')

# Sanity round-trip
loaded = joblib.load(MODEL_OUT)
print(f'Round-trip OK: {type(loaded).__name__} '
      f'| feats={len(loaded.feature_cols)} '
      f'| champion={loaded.champion_name} '
      f'| has_calibrator={loaded.score_calibrator is not None}')

In [ ]:
# ── ml_report_v3.json (métricas legíveis pelo /admin do bot) ─────────
report = {
    'schema_version':  3,
    'trained_at':      bundle.train_date,
    'champion':        CHAMPION_NAME,
    'n_features':      len(champion_feats),
    'feature_cols':    champion_feats,
    'momentum_feats':  MOMENTUM_FEATURES,
    'new_features_v31': NEW_FEATURES,
    'n_train':         int(len(X_full)),
    'horizon_days':    HORIZON_DAYS,
    'cv': {
        'kind':       'walk_forward_purged_expanding',
        'n_folds':    len(fold_specs),
        'purge_days': PURGE_DAYS,
    },
    'metrics': {
        'rho_alpha_mean': float(qualifiers.iloc[0]['rho_alpha_mean']),
        'rho_down_mean':  float(qualifiers.iloc[0]['rho_down_mean']),
        'topk_pnl_mean':  float(qualifiers.iloc[0]['topk_pnl_mean']),
        'brier_oof':      float(brier),
        'win_rate_alpha': float((df_v31['alpha_60d'] > 0.05).mean()),
    },
    'comparison': summary.round(4).to_dict(orient='records'),
    'target': {
        'name':    'alpha_60d',
        'formula': 'max_return_60d - spy_max_return_60d',
        'horizon_days': HORIZON_DAYS,
    },
}

with open(REPORT_OUT, 'w') as f:
    json.dump(report, f, indent=2)
print(f'Report escrito: {REPORT_OUT}')
print(json.dumps({k: v for k, v in report['metrics'].items()}, indent=2))

## 9. Deploy — `/admin_load_models`

Há duas formas de fazer o deploy do bundle para o Railway:

**Opção A — git push do .pkl directamente**
```bash
git add dip_models_v3.pkl ml_report_v3.json
git commit -m "ml(v3.1): retrain alpha_60d, 10 folds purged, +4 features, isotonic"
git push
```
O Railway redeployará automaticamente. Não precisa de `/admin_load_models`.

**Opção B — release no GitHub + `/admin_load_models`**

(Útil se quiseres reverter rápido para uma versão anterior sem rebuild.)

In [ ]:
# ── (Opcional) Upload via GitHub API + URL para o /admin_load_models ───
# Descomenta e cola o teu PAT (ghp_...) abaixo. Senão, salta esta célula
# e usa a opção A (git push directo).

import getpass, requests

UPLOAD_VIA_API = False   # ← muda para True se quiseres upload programático

if UPLOAD_VIA_API:
    PAT  = getpass.getpass('GitHub PAT (ghp_...): ').strip()
    OWNER, REPO = 'romeurf', 'DipRadar'
    TAG  = f'models-{datetime.utcnow().strftime("%Y-%m-%d")}-v31'
    HDRS = {'Authorization': f'token {PAT}', 'Accept': 'application/vnd.github+json'}

    # Get-or-create release
    r = requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/releases/tags/{TAG}', headers=HDRS)
    if r.status_code == 404:
        r = requests.post(
            f'https://api.github.com/repos/{OWNER}/{REPO}/releases',
            headers=HDRS,
            json={'tag_name': TAG, 'name': f'Models {TAG}',
                  'body': f'champion={CHAMPION_NAME} '
                          f'rho_alpha={qualifiers.iloc[0]["rho_alpha_mean"]:.3f} '
                          f'topk_pnl={qualifiers.iloc[0]["topk_pnl_mean"]:.3f}'},
        )
    r.raise_for_status()
    release = r.json()
    upload_url = release['upload_url'].split('{')[0]
    print(f'Release: {release["html_url"]}')

    # Apagar assets antigos com mesmo nome
    for a in release.get('assets', []):
        if a['name'] in ('dip_models_v3.pkl', 'ml_report_v3.json'):
            requests.delete(
                f'https://api.github.com/repos/{OWNER}/{REPO}/releases/assets/{a["id"]}',
                headers=HDRS,
            ).raise_for_status()

    asset_urls = []
    for path, ctype in [(MODEL_OUT, 'application/octet-stream'),
                        (REPORT_OUT, 'application/json')]:
        with open(path, 'rb') as f:
            r = requests.post(
                f'{upload_url}?name={path.name}',
                headers={**HDRS, 'Content-Type': ctype},
                data=f.read(),
            )
        r.raise_for_status()
        asset_urls.append(r.json()['browser_download_url'])

    print('\n🟢 No Telegram cola exactamente:')
    print(f'/admin_load_models {asset_urls[0]} {asset_urls[1]}')
else:
    print('UPLOAD_VIA_API=False — usa git push directo (opção A acima).')

## 10. Próximos passos

Não incluído neste retreino, mas natural a seguir:

1. **Validar live no bot** — após `/admin_load_models`, verificar `/admin`
   mostra `champion=XGB-alpha-v31`, `rho_alpha`, calibrator presente.
2. **Sector como feature ordinal** — single-encoding consistente entre
   treino e inferência.
3. **PIT fundamentals** — backfill histórico de `pe_ratio`, `short_interest`,
   `insider_buying` (precisa de fonte além de yfinance).
4. **SHAP explainer** — re-anexar ao bundle para destrancar
   `extract_shap_top3` no `ml_engine.py`.
5. **Allocation engine Fase 1** — motor read-only que envia plano
   mensal ao Telegram (ver `docs/allocation_engine_design.md`).